1. Fetch the artifact we just created (sample.csv) from W&B and read it with pandas:

In [ ]:
import wandb
import pandas as pd
# Note that we use save_code=True in the call to wandb.init so the notebook is uploaded and versioned by W&B
run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)

2. Explore the data in df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.head()

3. What do you notice in the data? Look around and see what you can find.




These columns are missing values: name, host_name, last_review, and reviews_per_month. The last_view column is shown as a date but in string format. The price column has a minimum of $0 and a maximum of $10,000, which seems unlikely. I have also noticed that under the name column there is not a standard of how they are presented. The last_review and reviews_per_month both have NaN values. The column minimum_nights also seems to have outliers due to the stay having 1250 nights under the observation of short-term rentals. 

4. Fix some of the little problems we have found in the data with the following code:

In [ ]:
# Drop outliers
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()
# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

In [ ]:
for col in ["name", "host_name", "neighbourhood_group", "neighbourhood"]:
    df[col] = df[col].astype(str).str.strip().str.lower()

In [ ]:
# Cap extreme minimum_nights values at 30 days
df = df[df["minimum_nights"] <= 30].copy()

Note how we did not impute missing values. We will do that in the inference pipeline, so we will be able to handle missing values also in production.

5. Check with df.info() that all obvious problems have been solved

In [ ]:
df.info()

6. Terminate the run by running `run.finish()`

In [ ]:
run.finish()

7. Save the notebook.